# Customer RFM Segmentation — SQL

SQL-cell equivalent of `customer_rfm_segmentation.ipynb`, using `%%sql`
cells (`NTILE` window functions) instead of the DataFrame API. Same
result: reads `retail.orders` and persists
`retail.customer_rfm_segments_v1`.

Requires the bronze table to already exist — run
`data-generators/retail_order_ingest.ipynb` first.

`%%sql` here follows Databricks-style notebook cell magic. If running
outside a platform that provides it natively, install and load
`sparksql-magic` first: `pip install sparksql-magic` then
`%load_ext sparksql_magic` (bound to the `spark` session created below).

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Transform + persist: RFM segments

Cleans cancellations/non-positive rows, computes recency/frequency/
monetary per customer, scores each into quintiles via `NTILE(5)`, then
labels the combined score into a named segment.

In [ ]:
%%sql
CREATE OR REPLACE TABLE retail.customer_rfm_segments_v1 AS
WITH clean AS (
    SELECT *
    FROM retail.orders
    WHERE invoice_no NOT LIKE 'C%'
      AND customer_id IS NOT NULL
      AND quantity > 0
      AND unit_price > 0
),
rfm AS (
    SELECT
        customer_id,
        FIRST(country) AS country,
        DATEDIFF((SELECT MAX(invoice_date) FROM clean), MAX(invoice_date)) AS recency_days,
        COUNT(DISTINCT invoice_no) AS frequency,
        ROUND(SUM(quantity * unit_price), 2) AS monetary
    FROM clean
    GROUP BY customer_id
),
scored AS (
    SELECT
        *,
        6 - NTILE(5) OVER (ORDER BY recency_days ASC) AS recency_score,
        NTILE(5) OVER (ORDER BY frequency ASC) AS frequency_score,
        NTILE(5) OVER (ORDER BY monetary ASC) AS monetary_score
    FROM rfm
)
SELECT
    customer_id,
    country,
    recency_days,
    frequency,
    monetary,
    recency_score,
    frequency_score,
    monetary_score,
    (recency_score + frequency_score + monetary_score) AS rfm_score,
    CASE
        WHEN (recency_score + frequency_score + monetary_score) >= 13 THEN 'Champions'
        WHEN (recency_score + frequency_score + monetary_score) >= 10 THEN 'Loyal Customers'
        WHEN (recency_score + frequency_score + monetary_score) >= 7 THEN 'Potential Loyalists'
        WHEN (recency_score + frequency_score + monetary_score) >= 4 THEN 'At Risk'
        ELSE 'Lost'
    END AS segment
FROM scored
ORDER BY rfm_score DESC

## Verify

In [ ]:
%%sql
SELECT COUNT(*) AS total_customers FROM retail.customer_rfm_segments_v1

In [ ]:
%%sql
SELECT segment, COUNT(*) AS customers
FROM retail.customer_rfm_segments_v1
GROUP BY segment
ORDER BY customers DESC